In [1]:
import numpy as np
from scipy.signal import place_poles
import numpy.typing as npt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize, NonlinearConstraint
N = 6
observable = False

def init_system(N:int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    A = np.eye(N, k=-1)
    A[:, -1] = np.random.randint(-N*2, -1, N)
    B = np.random.randint(-N, N, N).reshape(N,1)
    C = np.eye(1,N, k=N-1)
    return A, B, C

def get_observ_params(A: np.ndarray, C: np.ndarray, N :int) -> np.ndarray:
    desired_poles = np.random.choice(np.arange(-15, -6, 1), N, replace=False)
    res = place_poles(A.T, C.T, desired_poles)
    return res.gain_matrix

# def f(N) -> npt.NDArray:
#         return np.round(
            
#             np.random.randint(-N*3, N*3,N)
#             + np.random.random(N),
#             3,
#         ).reshape(1,N)
def f(N) -> npt.NDArray:
        l_true = -np.random.exponential(2, N-2)
        g_coeffs = np.random.normal(0, 3, N-2) 
        t = np.arange(N-1)
        powers = np.arange(N)
        basis_matrix = np.array([lt**powers for lt in l_true]).T
        f_true = basis_matrix @ g_coeffs
        f_noisy = f_true + np.random.normal(0, 1, N)
        return f_noisy.reshape(1,N)

A, B, C = init_system(N)
while not observable:
    Obsv = np.vstack([ C @ np.linalg.matrix_power(A, i) for i in range(N)])
    if np.linalg.matrix_rank(Obsv) < A.shape[0]:
        print(Obsv)
        print(np.linalg.matrix_rank(Obsv))
        A, B, C = init_system(N)
    else:
        observable = True

stable = False
F = f(N)
k_sigma = 1.5
while not stable:
    A_cl = A - k_sigma * B @ F
    if np.all(np.real(np.linalg.eigvals(A_cl)) < 0):
        stable = True
    else:
        # print(F)
        F = f(N)    
L = get_observ_params(A, C, N)

In [2]:
# A, B, C, F, L, k_sigma, N = (np.array([[ 0.,  0.,  0.,  0.,  0., -2.],
#         [ 1.,  0.,  0.,  0.,  0., -4.],
#         [ 0.,  1.,  0.,  0.,  0., -6.],
#         [ 0.,  0.,  1.,  0.,  0., -8.],
#         [ 0.,  0.,  0.,  1.,  0., -6.],
#         [ 0.,  0.,  0.,  0.,  1., -3.]]),
#  np.array([[ 4],
#         [ 4],
#         [ 2],
#         [-2],
#         [ 5],
#         [-1]]),
#  np.array([[0., 0., 0., 0., 0., 1.]]),
#  np.array([[ 13.422,  12.56 ,  -1.103, -14.334,   7.709, -16.452]]),
#  np.array([[1.03487800e+06, 6.34923997e+05, 1.60437999e+05, 2.13719999e+04,
#          1.57900000e+03, 5.89999999e+01]]),
#  1.5,
#  6)

In [3]:
def wandermond_lambdas_matrix(lambdas: npt.NDArray, n: int) -> npt.NDArray:
    return np.vstack([np.array(lambdas)**i for i in range(n)]).T

def wandermond_matrix(gammas: npt.NDArray, lambdas: npt.NDArray, n: int) -> npt.NDArray:
    return np.vstack([np.array(gammas) * np.array(lambdas)**i for i in range(n)]).T

def pairwise_cosines(x, K, N):
    if K == 1: return 0
    gammas, lambdas = x[:K], x[K:]
    V = wandermond_lambdas_matrix( lambdas, N)      # shape (K, N)
    norms = np.linalg.norm(V, axis=1, keepdims=True)
    if np.any(norms < 1e-12):
        # возвращаем большие значения, чтобы constraint точно нарушался
        m = K * (K - 1) // 2
        return np.full(m, 1e6)
    Vn = V / norms
    G = Vn @ Vn.T
    vals = []
    for i in range(K):
        for j in range(i + 1, K):
            vals.append(G[i, j])   # cos(v_i, v_j)
    return np.array(vals)


def approximation_elemnet(
    gammas: npt.NDArray, lambdas: npt.NDArray, n: int
) -> npt.NDArray:
    return np.sum(wandermond_matrix(gammas, lambdas, n), axis=0)


def functional_J(
    f: npt.NDArray, gammas: npt.NDArray, lambdas: npt.NDArray, n: int
) -> float:
    diff = (f - approximation_elemnet(gammas, lambdas, n)).flatten()
    return diff @ diff

def x0(K: int, lambdas: list[float] | None, gammas: list[float] | None) -> npt.NDArray:
    if lambdas is not None and gammas is not None:
        return gammas + [1.0] + lambdas + [- 0.1]    
    k_range = np.arange(1,K+1, 1)
    lambdas = -0.1*k_range
    gammas= k_range
    return np.concatenate([gammas, lambdas])

def approximation_F(F: npt.NDArray, K:int, N: int, lambdas: list[float] = None, gammas: list[float] = None) -> npt.NDArray:
    if lambdas is None: lambdas = []
    if gammas is None: gammas = []
    r =  F.flatten() if len(lambdas) == 0 else (F - approximation_elemnet(gammas, lambdas, N)).flatten()
    
    def lambda_score(l: float, r: npt.NDArray) -> float:
        v = np.array([l**i for i in range(N)]).flatten()
        return -((r @ v) ** 2) / (v @ v)

    b_lambda = [-1e9, -1e-3]

    solution = minimize(
        lambda x: lambda_score(x, r),
        x0=-0.1,
        bounds=[b_lambda],
        method="Nelder-Mead",
    )
    
    new_lambda = solution.x[0]
    print(new_lambda)
    lambdas.append(new_lambda)
    lambdas = np.array(lambdas)
    V = wandermond_lambdas_matrix(lambdas, N)
    gammas = np.linalg.solve(V @ V.T, V @ F.flatten())
    return  gammas,lambdas



In [4]:
def run_system(A: np.ndarray, B: np.ndarray, F: np.ndarray, N: int, dt:float=0.1, T: int =100) -> np.ndarray:
    def system_dynamics(t: float, x: np.ndarray) -> np.ndarray:
        u = -k_sigma * F @ x
        x_dot = A @ x + B @ u
        return x_dot.flatten()
    
    x0 = np.ones(N)
    t = np.arange(0, T, dt)
    sol = solve_ivp(system_dynamics, [0, t[-1]], x0, t_eval=t)
    sigma_ideal = (F @ sol.y).flatten()
    u_ideal = -k_sigma * sigma_ideal.flatten()
    return  u_ideal, sigma_ideal

u_ideal, sigma_ideal = run_system(A, B, F, N, 0.1, 100)
# u_ideal, sigma_ideal

In [5]:
def run_full_observe(A: np.ndarray, B: np.ndarray, C: np.ndarray, L: np.ndarray, F: np.ndarray, N: int, dt:float=0.1, T: int =100) -> np.ndarray:
    def system_dynamics(t: float, x: np.ndarray) -> np.ndarray:
        x_true, x_observe = x[:N], x[N:]
        u = -k_sigma * F @ x_observe
        dx_true = A @ x_true + B @ u
        dx_observe =  A @ x_observe + B @ u + (L * (C @ x_true - C @ x_observe)).flatten()
        return np.concatenate([dx_true, dx_observe]).flatten()
    
    x0 = np.concatenate([np.ones(N), np.zeros(N)])
    t = np.arange(0, T, dt)
    sol = solve_ivp(system_dynamics, [0, t[-1]], x0, t_eval=t)
    x_true, x_observe = sol.y[:N], sol.y[N:] 
    sigma_full = (F @ x_observe).flatten()   
    u_full = -k_sigma * sigma_full.flatten()
    
    return  u_full, sigma_full

u_full, sigma_full = run_full_observe(A, B, C, L, F, N, 0.1, 100)
# u_full, sigma_full

In [6]:
g_lambdas, g_gammas = None, None


def run_func_observe(A: np.ndarray, B: np.ndarray, C: np.ndarray, L: np.ndarray, F: np.ndarray,K: int, N: int, dt:float=0.1, T: int =100) -> np.ndarray:
    global g_lambdas, g_gammas
    gammas, lambdas = approximation_F(F, K, N, g_lambdas, g_gammas)
    g_lambdas, g_gammas = lambdas.tolist(), gammas.tolist()
    gamma_0 = (F[C==1] - gammas@(lambdas**(N-1))).item()
    print(f"MinJ: {functional_J(F, gammas, lambdas, N)}")
    print(lambdas, gammas, gamma_0)
    LAMBDAS = wandermond_lambdas_matrix(lambdas, N)
    H = (LAMBDAS@B).flatten()
    # print(H)
    G = np.concatenate([-pow_l @ (np.eye(N) * l - A)@C.T for pow_l , l in zip(LAMBDAS, lambdas)])
    # print(G)
    def system_dynamics(t: float, x: np.ndarray) -> np.ndarray:
        x_true, sigma = x[:N], x[N:]
        sigma_observe = gammas@sigma +gamma_0*C@x_true
        u = -k_sigma * sigma_observe
        # print(A @ x_true)
        # print(B * u)
        dx_true = A @ x_true + (B * u).flatten()
        # print(lambdas*  sigma_observe)
        # print(H*u)
        # print(G*(C @ x_true))
        dsigma_observe = lambdas* sigma + H*u + G*(C @ x_true)
        # print(lambdas @ (F @ x_observe))
        # print(sigma)
        # print(dx_true)
        # print(dsigma_observe)
        return np.concatenate([dx_true, dsigma_observe]).flatten()
    
    x0 = np.concatenate([np.ones(N), np.zeros(K)])
    t = np.arange(0, T, dt)
    sol = solve_ivp(system_dynamics, [0, t[-1]], x0, t_eval=t)
    x_true, sigma_observe = sol.y[:N], sol.y[N:]  
    y = (C @ x_true).flatten()  
    sigma_func = gammas@sigma_observe +gamma_0*y
    u_func = -k_sigma * sigma_func
    
    return  u_func, sigma_func

func_observer = {k: run_func_observe(A, B, C, L, F, k, N, 0.1, 100) for k in range(1, N)}

-0.16851562500000025
MinJ: 65.7338608073508
[-0.16851563] [5.01148294] 6.1937960704745
-0.001
MinJ: 65.36249154996206
[-0.16851563 -0.001     ] [ 8.54604862 -3.63725917] 6.19427639740109
-0.09031250000000002
MinJ: 55.00594615141826
[-0.16851563 -0.001      -0.0903125 ] [-229.1191827  -217.82196038  451.84939244] 6.16469384409609
-0.1357031250000001
MinJ: 33.37083871458089
[-0.16851563 -0.001      -0.0903125  -0.13570313] [-10487.85747949   2068.81289404 -13748.44854199  22172.40144693] 5.705653808406458
-0.11125000000000002
MinJ: 23.094806321351115
[-0.16851563 -0.001      -0.0903125  -0.13570313 -0.11125   ] [-132878.81844194  -12422.52547721  454521.1096158   644404.75028455
 -953619.60766826] 4.2713445922957085


In [7]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

dash_styles = ['dash', 'dot', 'dashdot', 'longdash', 'longdashdot', 'solid']

fig = go.Figure(
    layout=dict(
        legend=dict(
            yanchor="top",
            y=-0.2,
            xanchor="left",
            x=0.01,
            orientation="h",
        )
    )
)
fig.update_layout(
    xaxis_title="Время, t",
    yaxis_title="Управление, u",
    plot_bgcolor="white",
)
fig.update_xaxes(
    mirror=True,
    ticks="outside",
    showline=True,
    linecolor="black",
    gridcolor="lightgrey",
)
fig.update_yaxes(
    mirror=True,
    ticks="outside",
    showline=True,
    linecolor="black",
    gridcolor="lightgrey",
)
t=np.arange(0, 100, 0.1)
fig.add_trace(go.Scatter(x=t, y=u_ideal, name='Полный наблюдатель'),)
# fig.add_trace(go.Scatter( y=u_full, name='u_full'), row=1, col=1)
for k, (u_func, sigma_func) in func_observer.items():
    dash_с = dash_styles[k % len(dash_styles)]
    fig.add_trace(go.Scatter(x=t, y=u_func, name=f'Наблюдатель при k={k}', line=dict(dash=dash_с)), )

# fig.add_trace(go.Scatter( y=sigma_ideal, name='sigma_ideal'), row=2, col=1)
# fig.add_trace(go.Scatter( y=sigma_full, name='sigma_full'), row=2, col=1)
# for k, (u_func, sigma_func) in func_observer.items():
#     dash_с = dash_styles[k % len(dash_styles)]
#     fig.add_trace(go.Scatter(y=sigma_func, name=f'sigma_func_{k}', line=dict(dash=dash_с)), row=2, col=1)
fig.update_layout(height=600, width=1200)
fig.show()